In [1]:
import matplotlib.pyplot as plt

In [ ]:
# Visualization helper function
def plot_forecast(
    context_df: pd.DataFrame,
    pred_df: pd.DataFrame,
    test_df: pd.DataFrame,
    target_column: str,
    timeseries_id: str,
    id_column: str = "id",
    timestamp_column: str = "timestamp",
    history_length: int = 256,
    title_suffix: str = "",
):
    ts_context = context_df.query(f"{id_column} == @timeseries_id").set_index(timestamp_column)[target_column]
    ts_pred = pred_df.query(f"{id_column} == @timeseries_id and target_name == @target_column").set_index(
        timestamp_column
    )[["0.1", "predictions", "0.9"]]
    ts_ground_truth = test_df.query(f"{id_column} == @timeseries_id").set_index(timestamp_column)[target_column]

    last_date = ts_context.index.max()
    start_idx = max(0, len(ts_context) - history_length)
    plot_cutoff = ts_context.index[start_idx]
    ts_context = ts_context[ts_context.index >= plot_cutoff]
    ts_pred = ts_pred[ts_pred.index >= plot_cutoff]
    ts_ground_truth = ts_ground_truth[ts_ground_truth.index >= plot_cutoff]

    fig = plt.figure(figsize=(12, 3))
    ax = fig.gca()
    ts_context.plot(ax=ax, label=f"historical {target_column}", color="xkcd:azure")
    ts_ground_truth.plot(ax=ax, label=f"future {target_column} (ground truth)", color="xkcd:grass green")
    ts_pred["predictions"].plot(ax=ax, label="forecast", color="xkcd:violet")
    ax.fill_between(
        ts_pred.index,
        ts_pred["0.1"],
        ts_pred["0.9"],
        alpha=0.7,
        label="prediction interval",
        color="xkcd:light lavender",
    )
    ax.axvline(x=last_date, color="black", linestyle="--", alpha=0.5)
    ax.legend(loc="upper left")
    ax.set_title(f"{target_column} forecast for {timeseries_id} {title_suffix}")
    fig.show()

In [ ]:
def sherpa_get_feature(
    context_df: pd.DataFrame,
    resource_id: str,
    metricName: str,
):
    return context_df.loc[(context_df["item_id"] == resource_id) & (context_df["metricName"] == metricName)].copy()

def sherpa_combine_resources(
    context_df: pd.DataFrame,
    resource_id: str,
    target_metric: str
):

    resource_df = context_df.loc[context_df["item_id"] == resource_id]
    
    # Get all metric types for resource
    metric_types = resource_df["metricName"].unique()
    metric_types = np.delete(metric_types, np.where(metric_types == target_metric))
    
    target_df = resource_df.loc[resource_df["metricName"] == target_metric]
    
    for metric in metric_types:
        # construct merge df
        merge_df = resource_df.loc[resource_df["metricName"] == metric].drop(columns="metricName")
        merge_df = merge_df.rename(columns={"target": metric})
        # merge
        target_df = target_df.merge(merge_df, on=["item_id", "timestamp"])

    return target_df

def quantify_error_rmse(
    ground_truth_df: pd.DataFrame,
    prediction_df: pd.DataFrame
):
    rmse = np.sqrt(
        np.mean(
            (
                ground_truth_df["target"].to_numpy()
                - prediction_df["predictions"].to_numpy()
            ) ** 2
        )
    )
    print(f"Root Mean Squared Error: {rmse}")
    return rmse